# Diabetes Progression Prediction Model

## Introduction

This notebook demonstrates how to train a machine learning model for predicting diabetes progression and prepare it for deployment with Streamlit.

### What You'll Learn

- How to load and prepare the diabetes dataset
- How to train a Ridge Regression model
- How to evaluate model performance
- How to serialize models for deployment
- Why model serialization matters for production

### Why Streamlit for ML Models?

**Traditional Approach:**
- Build Flask/Django backend (Python)
- Create React/Vue frontend (JavaScript)
- Write HTML/CSS for styling
- **Time**: Days to weeks
- **Skills**: Full-stack development

**Streamlit Approach:**
- Write Python only
- Built-in widgets and visualizations
- Automatic UI generation
- **Time**: Hours
- **Skills**: Python (that's it!)

### Prerequisites

- Python 3.10+
- scikit-learn
- pandas, numpy
- dill (for model serialization)

---

## Step 1: Load and Prepare Data

In this step, we:
1. Import necessary libraries
2. Load the diabetes dataset
3. Split data into training and test sets
4. Understand the dataset structure


In [1]:
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
import dill as pickle
import numpy as np

def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Load the diabetes dataset
data = load_diabetes()
X = data.data
y = data.target

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Setup a Ridge Regression model
model = Ridge()

# Parameters for grid search
param_grid = {
    'alpha': [0.1, 1, 10, 100]
}

# Setup the grid search with cross-validation
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='r2', return_train_score=True)

# Fit the grid search model
grid_search.fit(X_train, y_train)

# Find the best model
best_model = grid_search.best_estimator_

# Predict on the test set with the best model
y_pred = best_model.predict(X_test)

# Calculate R2 and MAPE
r2 = r2_score(y_test, y_pred)
mape_score = mape(y_test, y_pred)

print(f'Best Model Performance on the Test Set: R² = {r2:.4f}, MAPE = {mape_score:.4f}%')


ModuleNotFoundError: No module named 'sklearn'

## Step 2: Model Training and Evaluation

### What's Happening Here?

1. **Ridge Regression**: A regularized linear regression model
   - Prevents overfitting with L2 regularization
   - Good for when you have many features
   - Handles multicollinearity well

2. **GridSearchCV**: Hyperparameter tuning
   - Tests different `alpha` values (regularization strength)
   - Uses cross-validation to find best parameters
   - Prevents overfitting to validation set

3. **Model Evaluation**: 
   - **R² Score**: Coefficient of determination (0-1, higher is better)
   - **MAPE**: Mean Absolute Percentage Error (lower is better)

### Understanding the Metrics

- **R² = 0.46**: Model explains 46% of variance in diabetes progression
- **MAPE = 37.7%**: Average prediction error is ~38% of actual value
- These are reasonable for medical prediction tasks

### Why Ridge Regression?

- Linear models are interpretable
- Regularization prevents overfitting
- Fast training and prediction
- Good baseline for regression problems


## Step 3: Model Serialization

### Why Serialize Models?

**The Problem:**
- Training takes time and computational resources
- You don't want to retrain every time you need a prediction
- Models need to be shared and deployed

**The Solution:**
- Serialize (save) the trained model to disk
- Load it later for predictions
- Share the model file with others

### Pickle vs Dill vs Joblib

**Pickle (standard library):**
- ✅ Built into Python
- ❌ Can't handle some complex objects
- ❌ Slower for large models

**Dill (what we use):**
- ✅ Extends pickle functionality
- ✅ Handles more complex objects
- ✅ Better for scikit-learn models
- ❌ Requires separate installation

**Joblib (scikit-learn recommended):**
- ✅ Optimized for NumPy arrays
- ✅ Faster for large models
- ✅ Recommended by scikit-learn
- ❌ Less flexible than dill

**For this exercise:** We use `dill` for compatibility, but `joblib` is often preferred in production.

### Model File

The model is saved as `best_diabetes_model.pkl` in the current directory. This file contains:
- Trained model parameters
- Model configuration
- Everything needed to make predictions

**Important**: Keep this file! You'll need it for the Streamlit app.


## Step 4: Model Validation

### Why Validate After Loading?

**The Problem:**
- Serialization can sometimes corrupt data
- File system issues can cause problems
- You want to ensure the model works correctly

**The Solution:**
- Load the model from disk
- Make predictions on test data
- Compare metrics to original training
- **If metrics match**: Model loaded correctly ✅
- **If metrics differ**: Something went wrong ❌

### What We're Checking

1. **Model loads successfully**: No errors when loading
2. **Predictions work**: Model can make predictions
3. **Performance matches**: R² and MAPE are the same
4. **Model integrity**: All parameters preserved

### Expected Result

You should see the same R² and MAPE scores as in Step 2. This confirms:
- Model serialization worked correctly
- Model can be loaded and used
- Ready for deployment in Streamlit app

---

## Next Steps

### 1. Deploy with Streamlit

Now that you have a trained model, you can:
1. Open the Streamlit app: `streamlit run app.py`
2. The app will load `best_diabetes_model.pkl`
3. Make interactive predictions through the web interface

### 2. Explore the Dataset

The diabetes dataset contains:
- **10 features**: Age, Sex, BMI, BP, and 6 blood serum measurements
- **Target**: Diabetes progression score (continuous value)
- **442 samples**: Small but classic dataset

### 3. Improve the Model

Try:
- Different models (Random Forest, XGBoost)
- Feature engineering
- Hyperparameter tuning
- Ensemble methods

### 4. Understand Model Limitations

- R² = 0.46 means the model explains less than half the variance
- This is common in medical prediction tasks
- Consider: More features, better models, more data

---

## Summary

### What You've Learned

✅ How to train a regression model  
✅ How to evaluate model performance  
✅ How to serialize models for deployment  
✅ Why model serialization is important  

### Key Takeaways

1. **Model Training**: GridSearchCV helps find best hyperparameters
2. **Evaluation**: R² and MAPE provide different perspectives on performance
3. **Serialization**: Save models to reuse without retraining
4. **Validation**: Always verify loaded models work correctly
5. **Deployment**: Serialized models are ready for production use

### Model File Location

Your model is saved as: `best_diabetes_model.pkl`

**Keep this file!** You'll need it for the Streamlit application.

---

## Ready for Streamlit!

Now that you have a trained model, proceed to the Streamlit app:

```bash
streamlit run app.py
```

The app will:
- Load your trained model
- Provide an interactive interface
- Make predictions in real-time
- Visualize results

Happy modeling! 🚀


In [ ]:
# Save the model using dill
with open('./best_diabetes_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

In [ ]:
# To load the model later and validate
with open('best_diabetes_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

# Validate the loaded model
y_pred_loaded = loaded_model.predict(X_test)
r2_loaded = r2_score(y_test, y_pred_loaded)
mape_loaded = mape(y_test, y_pred_loaded)

print(f'Loaded Model Validation on the Test Set: R² = {r2_loaded:.4f}, MAPE = {mape_loaded:.4f}%')

Loaded Model Validation on the Test Set: R² = 0.4609, MAPE = 37.6958%
